# 01 · EDA y recorrido por el pipeline — Estadísticas de YouTube

Este notebook ejecuta el pipeline **etapa por etapa**, para ver qué hace cada una, con qué librerías y con qué parámetros. Antes de cada celda de código hay una celda de documentación con la tabla de funciones y parámetros, la documentación oficial y sugerencias para experimentar.

| Sección | Etapa | Actividad del taller |
| --- | --- | --- |
| 3 | Ingesta | 2.2 Acceso a los datos |
| 4 | Diccionario de datos | 2.1 Diccionario de datos |
| 5 | Validación y calidad | 3.1 Reporte de calidad |
| 6 | Limpieza del texto | 4.1 Mejoras |
| 7 | Features por video | 5.1 Variable objetivo |
| 8 | Distribuciones | 3.1 Calidad, 5.1 Variable objetivo |
| 9 | Análisis estadísticos | Análisis no triviales |
| 10 | Modelos de referencia | 5.1 Baseline y suficiencia |

**Antes de ejecutar nada:** *Archivo → Guardar una copia en Drive*. La copia es la que el equipo edita y entrega.

Ejecutar primero las celdas 1 y 2 y después las secciones en orden: cada una usa los resultados de la anterior.

### Celda 1 · Descargar el código del taller

**Qué hace:** descarga el código del taller desde GitHub en `/content/youtube-stats-nlp`, instala las librerías que Colab no trae y permite importar el paquete `ytnlp`. No hay que llenar nada.

| Herramienta | Uso en la celda | Parámetro o valor | Qué significa |
| --- | --- | --- | --- |
| `git clone` | Descarga el repositorio | `--depth 1` | Solo la última versión, sin el historial (más rápido) |
| `git clone` | | `-b main` | Rama que se descarga |
| `git fetch` + `git reset --hard` | Actualiza si ya se descargó en la sesión | `FETCH_HEAD` | Deja el código igual a la última versión publicada |
| `subprocess.run` | Ejecuta comandos del sistema desde Python | `capture_output=True`, `text=True` | Guarda la salida del comando como texto para mostrar errores |
| `pip install` | Instala dependencias | `-q -r requirements-colab.txt` | Modo silencioso; instala la lista del archivo |
| `sys.path.insert` | Hace importable el paquete | `0, ".../src"` | Python busca módulos primero en esa carpeta |

**Documentación oficial**

- git clone: https://git-scm.com/docs/git-clone
- subprocess: https://docs.python.org/3/library/subprocess.html
- pip install: https://pip.pypa.io/en/stable/cli/pip_install/
- sys.path: https://docs.python.org/3/library/sys.html#sys.path

In [ ]:
# @title 1 · Descargar el código del taller e instalar dependencias
# Repositorio público del taller (lo define el docente; no se modifica).
REPO_URL = "https://github.com/USUARIO/REPOSITORIO.git"
BRANCH = "main"

import os, subprocess, sys

def _git(*args):
    r = subprocess.run(["git", *args], capture_output=True, text=True)
    if r.returncode != 0:
        raise RuntimeError("git " + args[0] + " falló:\n" + r.stderr[-500:])

try:
    import google.colab  # noqa: F401
    EN_COLAB = True
except ImportError:
    EN_COLAB = False

if EN_COLAB:
    if "USUARIO/REPOSITORIO" in REPO_URL:
        raise ValueError("El repositorio del taller no está configurado; avisar al docente.")
    REPO_DIR = "/content/youtube-stats-nlp"
    if os.path.exists(REPO_DIR):  # actualiza a la última versión publicada
        _git("-C", REPO_DIR, "fetch", "-q", "--depth", "1", "origin", BRANCH)
        _git("-C", REPO_DIR, "reset", "-q", "--hard", "FETCH_HEAD")
    else:
        _git("clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r",
                    f"{REPO_DIR}/requirements-colab.txt"], check=True)
    os.chdir(REPO_DIR)
else:  # computador local: el notebook se abre desde la carpeta notebooks/
    REPO_DIR = os.path.abspath("..") if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
    os.chdir(REPO_DIR)

sys.path.insert(0, os.path.join(REPO_DIR, "src"))
print("Código del taller listo en", REPO_DIR)

### Celda 2 · Montar Google Drive y cargar los secretos

**Qué hace:** conecta Google Drive (donde se guardan datos, reportes y configuración), lee las credenciales de los Secretos de Colab y muestra un resumen del entorno.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `google.colab.drive.mount` | `mountpoint` | `"/content/drive"` | Carpeta donde aparece Drive dentro de Colab |
| `google.colab.userdata.get` | nombre del secreto | `KAGGLE_API_TOKEN`, `YOUTUBE_API_KEY`, ... | Lee el valor guardado en el panel *Secretos*; se copia a `os.environ` |
| `os.environ` | `YTNLP_STORAGE` | ruta de `CARPETA_DRIVE` | Variable de entorno que indica al código dónde guardar resultados |
| `logging.basicConfig` | `level` | `logging.INFO` | Muestra los mensajes `INFO` y `WARNING` del pipeline |
| | `format` | `"%(levelname)s %(message)s"` | Formato de cada mensaje |
| | `force` | `True` | Reemplaza la configuración de registro que trae Colab |

**Parámetro del formulario:** `CARPETA_DRIVE` es la ruta, dentro de Drive, donde se guarda todo el trabajo del equipo. Si la carpeta compartida tiene otro nombre, se cambia aquí.

**Documentación oficial**

- Archivos y Google Drive en Colab: https://colab.research.google.com/github/googlecolab/colabtools/blob/main/notebooks/io.ipynb
- os.environ: https://docs.python.org/3/library/os.html#os.environ
- logging.basicConfig: https://docs.python.org/3/library/logging.html#logging.basicConfig

In [ ]:
# @title 2 · Montar Drive y cargar secretos
CARPETA_DRIVE = "MyDrive/ytnlp-proyecto"  # @param {type:"string"}

import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s %(message)s", force=True)

from ytnlp.colab import setup
info = setup(drive_folder=CARPETA_DRIVE)

## 3 · Ingesta

**Qué hace:** carga videos y comentarios de la fuente elegida y los convierte al esquema común del taller (nombres en `snake_case`, fechas en UTC, conteos numéricos).

**Parámetro del formulario:** `SOURCE` elige la fuente.

| Valor | Origen | Requisito |
| --- | --- | --- |
| `sample` | Muestra sintética incluida en el taller | Ninguno; sirve para practicar |
| `kaggle` | Dataset *YouTube Statistics* de Kaggle | Secreto `KAGGLE_API_TOKEN` |
| `api` | Última extracción de la YouTube Data API guardada en Drive | Haber ejecutado la sección 7 del notebook de inicio |

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `pd.read_csv` | ruta | `videos-stats.csv`, `comments.csv` | Lee los CSV de Kaggle |
| `DataFrame.rename` | `columns` | `{"Video ID": "video_id", ...}` | Pasa los nombres al esquema común |
| `pd.to_datetime` | `errors="coerce"`, `utc=True` | | Fechas inválidas pasan a `NaT`; todas quedan en UTC |
| `pd.to_numeric` | `errors="coerce"` | | Textos no numéricos pasan a `NaN` |
| `pd.read_parquet` | ruta | `data/raw/api/<fecha>/*.parquet` | Lee la extracción de la API (solo con `SOURCE = "api"`) |

**Documentación oficial**

- pandas.read_csv: https://pandas.pydata.org/docs/reference/api/pandas.read_csv.html
- pandas.DataFrame.rename: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.rename.html
- pandas.to_datetime: https://pandas.pydata.org/docs/reference/api/pandas.to_datetime.html
- pandas.read_parquet: https://pandas.pydata.org/docs/reference/api/pandas.read_parquet.html

In [ ]:
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
from IPython.display import display
from ytnlp.pipeline import ingest

SOURCE = "sample"  # @param ["sample", "kaggle", "api"]
videos, comments = ingest(SOURCE)
print("videos:", videos.shape, "| comentarios:", comments.shape)
display(videos.head(3)); display(comments.head(3))

## 4 · Diccionario de datos (Actividad 2.1)

**Qué hace:** describe cada columna con su tipo, porcentaje de nulos, número de valores distintos y un ejemplo. Es el punto de partida del informe.

| Función | Qué devuelve |
| --- | --- |
| `DataFrame.dtypes` | Tipo de dato de cada columna (`float64`, `object`, `datetime64[ns, UTC]`, ...) |
| `DataFrame.isna().mean()` | Proporción de valores nulos por columna; por 100 da el porcentaje |
| `DataFrame.nunique()` | Cantidad de valores distintos por columna (sin contar nulos) |
| `DataFrame.iloc[0]` | Primera fila, como ejemplo |

**Documentación oficial**

- pandas.DataFrame.isna: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.isna.html
- pandas.DataFrame.nunique: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.nunique.html

**Para experimentar:** agregar una columna `descripcion` al diccionario y completarla a mano para cada variable.

In [ ]:
def diccionario(df):
    return pd.DataFrame({
        "tipo": df.dtypes.astype(str),
        "% nulos": (100 * df.isna().mean()).round(2),
        "únicos": df.nunique(),
        "ejemplo": df.iloc[0],
    })

display(diccionario(videos)); display(diccionario(comments))

## 5 · Validación y calidad (Actividad 3.1)

**Qué hace:** aplica reglas de validación con **Pandera** y mide la calidad de los datos.

- **Reglas duras:** si una fila las viola, se elimina y se reporta (por ejemplo, sentimiento fuera de {0, 1, 2} o conteos negativos).
- **Reglas blandas:** solo se miden (por ejemplo, likes mayores que vistas); el equipo decide qué hacer.

**Reglas definidas en `validate.py`**

| Columna | Tipo | Regla | `nullable` | `coerce` |
| --- | --- | --- | --- | --- |
| `video_id` | texto | obligatoria | `False` | |
| `published_at` | fecha UTC | | `True` | `True` |
| `likes`, `comments`, `views` | número | `Check.ge(0)`: mayor o igual a 0 | `True` | `True` |
| `comment_likes` | número | `Check.ge(0)` | `True` | `True` |
| `sentiment` | número | `Check.isin([0, 1, 2])` | `True` | `True` |

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `pa.DataFrameSchema` | `columns` | diccionario de `pa.Column` | Conjunto de reglas de una tabla |
| | `strict` | `False` | Se permiten columnas que no están en el esquema |
| `pa.Column` | `nullable` | ver tabla | Si `False`, un nulo es un error |
| | `coerce` | `True` | Convierte al tipo indicado antes de validar |
| | `required` | `False` en columnas opcionales | La columna puede no existir |
| `pa.Check.ge` | `0` | | Valor mayor o igual a 0 |
| `pa.Check.isin` | `[0, 1, 2]` | | Valor dentro de la lista |
| `schema.validate` | `lazy` | `True` | Reúne todos los errores en lugar de detenerse en el primero |

**Documentación oficial**

- Pandera, esquemas de DataFrame: https://pandera.readthedocs.io/en/stable/dataframe_schemas.html
- Pandera, documentación general: https://pandera.readthedocs.io/en/stable/

**Para experimentar:** en una celda nueva, validar una copia de `videos` con un valor negativo en `views` y revisar `reglas_duras` en el resultado.

In [ ]:
from ytnlp.data.validate import validate

res = validate(videos, comments)
print("Filas eliminadas por reglas duras:")
print("  videos:", res.report["reglas_duras"]["videos"])
print("  comentarios:", res.report["reglas_duras"]["comentarios"])
pd.json_normalize(res.report, sep=" / ").T.rename(columns={0: "valor"})

## 6 · Limpieza del texto

**Qué hace:** normaliza cada comentario para que el análisis no dependa de detalles de escritura. La tabla muestra cada paso de `clean_text`, en orden.

| Paso | Función | Parámetro o patrón | Ejemplo |
| --- | --- | --- | --- |
| 1. Normalizar Unicode | `unicodedata.normalize` | `"NFKC"` | Unifica caracteres equivalentes (letras de ancho completo, ligaduras) |
| 2. URLs | `re.sub` | `https?://\S+` o `www\.\S+` → `<url>` | `https://x.co` → `<url>` |
| 3. Menciones | `re.sub` | `@\w+` → `<user>` | `@juan` → `<user>` |
| 4. Emojis | `emoji.demojize` | `delimiters=(" :", ": ")` | el emoji de fuego → `:fire:` |
| 5. Letras repetidas | `re.sub` | `(.)\1{3,}` → la letra 3 veces | `siiiiii` → `siii` |
| 6. Espacios y mayúsculas | `re.sub`, `str.lower` | `\s+` → un espacio | `Hola   MUNDO` → `hola mundo` |

Después, `clean_comments` elimina los comentarios vacíos (menos de `cleaning.min_comment_tokens` palabras) y los duplicados dentro de un mismo video (`drop_duplicates`).

**Por qué los emojis se convierten y no se borran:** expresan sentimiento. `:fire:` o `:thumbs_down:` aportan información que se perdería al eliminarlos.

**Documentación oficial**

- unicodedata.normalize: https://docs.python.org/3/library/unicodedata.html
- Expresiones regulares (re): https://docs.python.org/3/library/re.html
- emoji.demojize: https://carpedm20.github.io/emoji/docs/api.html
- pandas.DataFrame.drop_duplicates: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.drop_duplicates.html

**Para experimentar:** probar `clean_text` con comentarios propios en la primera línea de la celda; cambiar `demojize=False` y comparar.

In [ ]:
from ytnlp.data.clean import clean_comments, clean_text, clean_videos

ejemplo = "Me ENCANTAAAAA este video \U0001F525 https://x.co/abc @juan   !!!"
print("original:", ejemplo.encode("ascii", "backslashreplace").decode())
print("limpio:  ", clean_text(ejemplo, demojize=True))

v = clean_videos(res.videos)
c = clean_comments(res.comments)
print(f"comentarios: {len(res.comments)} antes de limpiar, {len(c)} después")
c[["comment_raw", "comment", "n_tokens"]].head(8)

## 7 · Features por video

**Qué hace:** construye una tabla con **una fila por video**. Los comentarios se resumen en features numéricas y se calculan las variables objetivo. La división entre entrenamiento y prueba se hará por video, nunca por comentario.

**Features de los comentarios**

| Feature | Cómo se calcula | Función |
| --- | --- | --- |
| `sent_mean` | Promedio del sentimiento con signo (−1 negativo, 0 neutral, +1 positivo) | `groupby(...).mean()` |
| `sent_weighted` | Promedio ponderado por likes: peso = log(1 + likes) + 1 | `np.average(weights=...)` |
| `sent_std` | Desviación estándar del sentimiento | `groupby(...).std()` |
| `polarization` | Entropía de la distribución de sentimiento, dividida por log(3): 0 = todos opinan igual, 1 = opiniones divididas por igual | `np.bincount`, `np.log` |
| `top_comment_sent` | Sentimiento del comentario con más likes | `idxmax` |
| `pct_pos`, `pct_neg` | Porcentaje de comentarios positivos y negativos | |
| `tokens_mean` | Palabras promedio por comentario | |
| `ttr` | *Type-token ratio*: palabras distintas / palabras totales (diversidad léxica) | |
| `pct_question`, `pct_exclaim` | Porcentaje de comentarios con `?` o `!` | `str.contains` |
| `emoji_per_comment` | Emojis por comentario (después de `demojize`) | `str.count` |
| `doc` | Todos los comentarios del video unidos en un texto | para TF-IDF |

**Features de contexto:** `keyword`, día y hora de publicación, edad del video en días (y su logaritmo), longitud del título y, con datos de la API, duración y suscriptores (en logaritmo).

**Variables objetivo** (nunca se usan como features)

| Variable | Fórmula | Función |
| --- | --- | --- |
| `target_engagement` | (likes + comentarios) / vistas, solo si vistas ≥ `min_views_for_engagement` (100) | |
| `target_log_views` | log(1 + vistas) | `np.log1p` |
| `target_log_likes` | log(1 + likes) | `np.log1p` |
| `target_perf_class` | bajo / medio / alto según los cuantiles 0,33 y 0,66 del engagement | `Series.quantile`, `pd.cut(bins, labels)` |

**Sentimiento de los datos de la API:** la API no trae etiqueta de sentimiento. En ese caso se usa un léxico mínimo de palabras positivas y negativas (`lexicon_sentiment`). Reemplazarlo por un modelo preentrenado es uno de los ejercicios de la guía (Parte 8).

**Documentación oficial**

- pandas.DataFrame.groupby: https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.groupby.html
- numpy.average: https://numpy.org/doc/stable/reference/generated/numpy.average.html
- numpy.log1p: https://numpy.org/doc/stable/reference/generated/numpy.log1p.html
- pandas.cut: https://pandas.pydata.org/docs/reference/api/pandas.cut.html

**Para experimentar:** comparar `sent_mean` con `sent_weighted`: ¿en qué videos difieren más? ¿Qué indica eso sobre los comentarios con más likes?

In [ ]:
from ytnlp.features.build_features import build

feats = build(v, c)
print("tabla de features:", feats.shape)
cols = ["video_id", "keyword", "sent_mean", "sent_weighted", "polarization", "ttr",
        "tokens_mean", "target_engagement", "target_perf_class"]
display(feats[cols].head(8))
display(feats["target_perf_class"].value_counts(dropna=False).rename("videos por clase"))

## 8 · Distribuciones clave

**Qué hace:** grafica las distribuciones de vistas, engagement y sentimiento, y el engagement por keyword. Se usa escala logarítmica porque las vistas tienen **cola pesada**: unos pocos videos concentran la mayoría de las vistas.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `plt.subplots` | `1, 3`, `figsize=(14, 3.5)` | | Una fila con tres gráficos; tamaño en pulgadas |
| `Series.hist` | `bins` | `40` | Número de barras del histograma |
| `np.log10` | | | Logaritmo base 10: cada unidad es un orden de magnitud |
| `Series.plot.bar`, `plot.barh` | | | Barras verticales y horizontales |

**Documentación oficial**

- matplotlib.pyplot.subplots: https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.subplots.html
- pandas.Series.hist: https://pandas.pydata.org/docs/reference/api/pandas.Series.hist.html
- Visualización con pandas: https://pandas.pydata.org/docs/user_guide/visualization.html

**Para experimentar:** quitar `np.log10` del primer gráfico y comparar: ¿qué se ve y qué se pierde?

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(14, 3.5))
np.log10(v["views"].dropna().clip(lower=1)).hist(bins=40, ax=ax[0]); ax[0].set_title("log10(vistas)")
np.log10(feats["target_engagement"].dropna()).hist(bins=40, ax=ax[1]); ax[1].set_title("log10(engagement)")
c["sentiment"].value_counts().sort_index().plot.bar(ax=ax[2]); ax[2].set_title("sentimiento (0 / 1 / 2)")
plt.tight_layout(); plt.show()

feats.groupby("keyword")["target_engagement"].median().sort_values().plot.barh(
    title="Engagement mediano por keyword"); plt.show()

## 9 · Análisis estadísticos no triviales

**Qué hace:** ejecuta los análisis uno por uno para ver sus parámetros y resultados. Cada función devuelve la pregunta, la prueba, el resultado y una conclusión automática que el equipo debe revisar y reescribir con sus palabras.

**Regla general:** un valor p pequeño indica que el efecto probablemente no se debe al azar, pero no dice si es grande. Por eso se reportan también **tamaños de efecto** e **intervalos de confianza**.

### 9.1 Correlación de rangos con intervalo por bootstrap

¿El sentimiento promedio se asocia con el engagement? Spearman y Kendall miden asociación monótona usando rangos, por lo que son robustos a valores extremos como los videos virales.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `scipy.stats.spearmanr` | `x`, `y` | `sent_mean`, `target_engagement` | Devuelve ρ (de −1 a 1) y su valor p |
| `scipy.stats.kendalltau` | `x`, `y` | | Devuelve τ, otra medida de asociación de rangos |
| bootstrap | `n_boot` | `stats.bootstrap_iterations` = 2000 | Se remuestrean los videos 2000 veces; el intervalo del 95 % va del percentil 2,5 al 97,5 |
| `np.random.default_rng` | `seed` | `42` | Generador aleatorio con semilla fija: resultados reproducibles |

Lectura: si el intervalo no incluye 0, hay asociación. Valores absolutos de ρ cercanos a 0,1 son débiles, a 0,3 moderados y a 0,5 fuertes.

- spearmanr: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.spearmanr.html
- kendalltau: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kendalltau.html
- Bootstrap en SciPy (alternativa lista para usar): https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.bootstrap.html

**Para experimentar:** cambiar `x` por `polarization` o `sent_weighted`, y bajar `n_boot` a 200 para ver cómo cambia el intervalo.

In [ ]:
import json
from ytnlp.analysis import stats as st

def mostrar(r):
    print("Pregunta:", r["pregunta"])
    print("Prueba:  ", r["prueba"])
    print("Resultado:", json.dumps(r["resultado"], indent=2, ensure_ascii=False, default=str))
    print("Conclusión:", r["conclusion"])

mostrar(st.spearman_bootstrap(feats, x="sent_mean", y="target_engagement", n_boot=2000))

### 9.2 Diferencias entre grupos: Kruskal-Wallis y Dunn

¿El engagement difiere entre keywords? Kruskal-Wallis compara la distribución de varios grupos sin suponer normalidad. Si es significativa, la prueba de Dunn indica qué pares difieren, con la corrección de Bonferroni para controlar los falsos positivos.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `scipy.stats.kruskal` | grupos | engagement de cada keyword con al menos 3 videos | Devuelve el estadístico H y el valor p |
| ε² (épsilon cuadrado) | | (H − k + 1) / (n − k) | Tamaño del efecto: 0,01 pequeño, 0,08 moderado, 0,26 grande |
| Dunn con Bonferroni | `alpha` | `stats.alpha` = 0,05 | Cada valor p se multiplica por el número de pares comparados |

- kruskal: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.kruskal.html
- Pruebas post hoc, incluida Dunn (scikit-posthocs): https://scikit-posthocs.readthedocs.io/en/latest/

Requiere al menos dos keywords con tres o más videos cada una; si no se cumple, el resultado lo indica.

In [ ]:
r = st._safe("4_engagement_por_keyword", st.kruskal_dunn, feats, "target_engagement", "keyword")
mostrar(r)

### 9.3 Leyes de Zipf y Heaps

¿El vocabulario de los comentarios se comporta como lenguaje natural?

- **Zipf:** la frecuencia de una palabra es aproximadamente proporcional a 1 / rango. En escala log-log es una recta con pendiente cercana a −1.
- **Heaps:** el vocabulario crece como V = K · N^β, con β típico entre 0,4 y 0,6.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `collections.Counter` | tokens | todas las palabras | Frecuencia de cada palabra |
| `np.polyfit` | `deg` | `1` | Ajusta una recta por mínimos cuadrados a log(frecuencia) vs log(rango) |
| | datos | la mitad más frecuente del vocabulario | Evita la cola de palabras que aparecen una sola vez (hapax) |

- numpy.polyfit: https://numpy.org/doc/stable/reference/generated/numpy.polyfit.html
- collections.Counter: https://docs.python.org/3/library/collections.html#collections.Counter
- Jurafsky y Martin, *Speech and Language Processing*, capítulo 2: https://web.stanford.edu/~jurafsky/slp3/

In [ ]:
r = st.zipf_heaps(c["comment"])
mostrar({k: val for k, val in r.items() if k != "_freq"})

freq = r["_freq"]
plt.loglog(np.arange(1, len(freq) + 1), freq)
plt.xlabel("rango"); plt.ylabel("frecuencia"); plt.title("Ley de Zipf en los comentarios"); plt.show()

### 9.4 Comentario más votado vs. promedio: Wilcoxon pareado

¿El sentimiento del comentario con más likes difiere del promedio del video? Es una comparación **pareada**: cada video aporta los dos valores.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `scipy.stats.wilcoxon` | `x`, `y` | `top_comment_sent`, `sent_mean` | Prueba de rangos con signo para datos pareados |
| | `zero_method` | `"zsplit"` | Las diferencias iguales a cero se reparten entre rangos positivos y negativos |
| correlación rango-biserial | | 1 − 2W / (n(n+1)/2) | Tamaño del efecto, entre −1 y 1 |

- wilcoxon: https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.wilcoxon.html

In [ ]:
mostrar(st._safe("8_comentario_top_vs_promedio", st.top_vs_mean, feats))

### 9.5 Información mutua con prueba de permutación

¿Cuánta información aporta cada feature de los comentarios sobre la clase de desempeño (bajo, medio, alto)? A diferencia de la correlación, la información mutua detecta relaciones no lineales.

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `sklearn.feature_selection.mutual_info_classif` | `X`, `y` | features de comentarios, `target_perf_class` | Estima la información mutua (en nats) de cada feature con la clase |
| | `n_neighbors` | 3 (por defecto) | Vecinos usados en el estimador; más vecinos reducen la varianza |
| | `random_state` | `42` | Resultados reproducibles |
| prueba de permutación | `n_perm` | `200` | Se baraja la clase 200 veces; `p_perm` es la proporción de veces que el azar iguala o supera el valor real |

- mutual_info_classif: https://scikit-learn.org/stable/modules/generated/sklearn.feature_selection.mutual_info_classif.html

Los demás análisis (cola pesada, polarización, regresión con controles y palabras distintivas) están en `reports/stats_report.md`, y su interpretación en la Parte 7 de la guía.

In [ ]:
r = st._safe("9_informacion_mutua", st.mutual_information,
             feats.dropna(subset=["target_perf_class"]), st.COMMENT_FEATS, "target_perf_class", n_perm=200)
mostrar(r)

## 10 · Modelos de referencia y curva de aprendizaje

**Qué hace:** entrena cuatro modelos sencillos y los compara en un conjunto de prueba. Responde dos preguntas: **¿los comentarios aportan?** (comparar el modelo sin texto con los que usan comentarios) y **¿los datos son suficientes?** (curva de aprendizaje).

| Modelo | Qué usa |
| --- | --- |
| `dummy` | Nada: predice siempre el promedio o la clase más frecuente. Es el piso mínimo |
| `contexto (sin texto)` | keyword, día y hora de publicación, edad del video, características del título |
| `contexto + features de comentarios` | Lo anterior más sentimiento, polarización, longitud y diversidad léxica |
| `contexto + features + TF-IDF` | Lo anterior más las palabras de los comentarios |

**Preprocesamiento** (`ColumnTransformer` aplica un tratamiento distinto a cada tipo de columna)

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `SimpleImputer` | `strategy` | `"median"` | Rellena nulos numéricos con la mediana, solo dentro del modelo |
| `StandardScaler` | | | Centra cada variable en 0 con desviación 1 |
| `OneHotEncoder` | `handle_unknown` | `"ignore"` | Una keyword nueva en la prueba se codifica con ceros en lugar de producir un error |
| `TfidfVectorizer` | `min_df` | `2` | Ignora palabras que aparecen en menos de 2 videos |
| | `max_features` | `20000` | Conserva las 20.000 palabras o pares de palabras más frecuentes |
| | `ngram_range` | `(1, 2)` | Usa palabras sueltas y pares de palabras consecutivas |
| | `sublinear_tf` | `True` | Usa 1 + log(frecuencia), para que las palabras repetidas no dominen |

**Modelos y evaluación**

| Función | Parámetro | Valor usado | Qué significa |
| --- | --- | --- | --- |
| `Ridge` (regresión) | `alpha` | `1.0` | Fuerza de la regularización L2: más alto, coeficientes más pequeños |
| `LogisticRegression` (clasificación) | `C` | `1.0` | Inverso de la regularización: más bajo, regularización más fuerte |
| | `max_iter` | `2000` | Iteraciones máximas del optimizador (el valor por defecto, 100, puede no converger con TF-IDF) |
| `DummyRegressor` / `DummyClassifier` | `strategy` | promedio / `"most_frequent"` | Predicción constante de referencia |
| `train_test_split` | `test_size` | `0.2` | 20 % de los videos para la prueba |
| | `random_state` | `42` | División reproducible |
| | `stratify` | la clase (solo en clasificación) | Mantiene la proporción de clases en entrenamiento y prueba |
| Métricas de regresión | | MAE, R², Spearman | Error absoluto medio, varianza explicada, correlación de rangos entre real y predicho |
| Métricas de clasificación | | accuracy, F1 macro | Aciertos; promedio del F1 de cada clase (no favorece a la clase mayoritaria) |

Con la variable `engagement`, el modelo predice log(engagement), que tiene una distribución más simétrica. Se necesitan al menos 30 videos con la variable objetivo.

**Documentación oficial**

- ColumnTransformer: https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html
- SimpleImputer: https://scikit-learn.org/stable/modules/generated/sklearn.impute.SimpleImputer.html
- StandardScaler: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html
- OneHotEncoder: https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html
- TfidfVectorizer: https://scikit-learn.org/stable/modules/generated/sklearn.feature_extraction.text.TfidfVectorizer.html
- Ridge: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html
- LogisticRegression: https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.LogisticRegression.html
- DummyClassifier: https://scikit-learn.org/stable/modules/generated/sklearn.dummy.DummyClassifier.html
- train_test_split: https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.train_test_split.html
- Métricas de evaluación: https://scikit-learn.org/stable/modules/model_evaluation.html
- Curvas de aprendizaje: https://scikit-learn.org/stable/modules/learning_curve.html
- Errores comunes, incluida la fuga de información: https://scikit-learn.org/stable/common_pitfalls.html

**Para experimentar:** ejecutar `baseline.run(feats, target="perf_class")` en una celda nueva y comparar; luego cambiar la variable en la sección 4 del notebook de inicio.

In [ ]:
from ytnlp.models import baseline

try:
    b = baseline.run(feats)
except ValueError as err:
    print("No se calcularon los modelos:", err)
else:
    print("Variable objetivo:", b["target"], "|", b["tipo"], "| train:", b["n_train"], "| test:", b["n_test"])
    display(pd.DataFrame(b["resultados"]).T)
    curva = pd.DataFrame(b["curva_aprendizaje"])
    display(curva)
    curva.plot(x="n_train", y=["train", "test"], marker="o",
               title=f"Curva de aprendizaje ({b['metrica_principal']})"); plt.show()
    print("Diagnóstico:", b["diagnostico_suficiencia"])

**Cómo leer la curva de aprendizaje**

- Si la métrica de prueba sigue subiendo al final, conseguir más videos mejoraría el modelo.
- Si la curva se aplana, más filas ayudan poco: conviene invertir en mejores features o mejores etiquetas.
- Si la diferencia entre entrenamiento y prueba es grande, el modelo se sobreajusta.

## Conclusiones del equipo

- Calidad de los datos:
- Suficiencia (según la curva de aprendizaje):
- ¿Los comentarios aportan información sobre el desempeño?
- Mejoras priorizadas:
- Variable objetivo elegida y por qué:
- Tres análisis estadísticos más relevantes y su interpretación: